In [1]:
from __future__ import print_function
import keras
import tensorflow as tf
from keras.layers import Dense, Conv2D, BatchNormalization, Activation
from keras.layers import AveragePooling2D, Input, Flatten
from tensorflow.keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras.callbacks import ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator
from keras.regularizers import l2
from keras import backend as K
from keras.models import Model
from keras.datasets import cifar100
from keras.utils.np_utils import to_categorical
import numpy as np
import os
from keras.layers import Conv2D,Conv1D,Reshape,MaxPooling2D,Multiply,GlobalAveragePooling2D,Multiply,Lambda, Activation,GlobalMaxPooling2D,Dense,Input

In [2]:

# Training parameters
#Ref: we use baseline resnet-56 code from https://github.com/mvaldenegro/paper-subensembles-image-classification/blob/master/ResNet.pybatch_size = 32 
batch_size = 32  # orig paper trained all networks with batch_size=128
epochs = 200
data_augmentation = True
num_classes = 100

# Subtracting pixel mean improves accuracy
subtract_pixel_mean = True

# Model parameter
# ----------------------------------------------------------------------------
#           |      | 200-epoch | Orig Paper| 200-epoch | Orig Paper| sec/epoch
# Model     |  n   | ResNet v1 | ResNet v1 | ResNet v2 | ResNet v2 | GTX1080Ti
#           |v1(v2)| %Accuracy | %Accuracy | %Accuracy | %Accuracy | v1 (v2)
# ----------------------------------------------------------------------------
# ResNet20  | 3 (2)| 92.16     | 91.25     | -----     | -----     | 35 (---)
# ResNet32  | 5(NA)| 92.46     | 92.49     | NA        | NA        | 50 ( NA)
# ResNet44  | 7(NA)| 92.50     | 92.83     | NA        | NA        | 70 ( NA)
# ResNet56  | 9 (6)| 92.71     | 93.03     | 93.01     | NA        | 90 (100)
# ResNet110 |18(12)| 92.65     | 93.39+-.16| 93.15     | 93.63     | 165(180)
# ResNet164 |27(18)| -----     | 94.07     | -----     | 94.54     | ---(---)
# ResNet1001| (111)| -----     | 92.39     | -----     | 95.08+-.14| ---(---)
# ---------------------------------------------------------------------------
n = 9 # For resnet56

# Model version
# Orig paper: version = 1 (ResNet v1), Improved ResNet: version = 2 (ResNet v2)
version = 1

# Computed depth from supplied model parameter n
if version == 1:
    depth = n * 6 + 2
elif version == 2:
    depth = n * 9 + 2

# Model name, depth and version
model_type = 'ResNet%dv%d' % (depth, version)

# Load the CIFAR10 data.
(x_train, y_train), (x_test, y_test) = cifar100.load_data()

# Input image dimensions.
input_shape = x_train.shape[1:]

# Normalize data.
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

# If subtract pixel mean is enabled
if subtract_pixel_mean:
    x_train_mean = np.mean(x_train, axis=0)
    x_train -= x_train_mean
    x_test -= x_train_mean

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')
print('y_train shape:', y_train.shape)

# Convert class vectors to binary class matrices.
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)


def lr_schedule(epoch):
    """Learning Rate Schedule

    Learning rate is scheduled to be reduced after 80, 120, 160, 180 epochs.
    Called automatically every epoch as part of callbacks during training.

    # Arguments
        epoch (int): The number of epochs

    # Returns
        lr (float32): learning rate
    """
    lr = 1e-3
    if epoch > 180:
        lr *= 0.5e-3
    elif epoch > 160:
        lr *= 1e-3
    elif epoch > 120:
        lr *= 1e-2
    elif epoch > 80:
        lr *= 1e-1
    print('Learning rate: ', lr)
    return lr


def resnet_layer(inputs,
                 num_filters=16,
                 kernel_size=3,
                 strides=1,
                 activation='relu',
                 batch_normalization=True,
                 conv_first=True):
    """2D Convolution-Batch Normalization-Activation stack builder

    # Arguments
        inputs (tensor): input tensor from input image or previous layer
        num_filters (int): Conv2D number of filters
        kernel_size (int): Conv2D square kernel dimensions
        strides (int): Conv2D square stride dimensions
        activation (string): activation name
        batch_normalization (bool): whether to include batch normalization
        conv_first (bool): conv-bn-activation (True) or
            bn-activation-conv (False)

    # Returns
        x (tensor): tensor as input to the next layer
    """
    conv = Conv2D(num_filters,
                  kernel_size=kernel_size,
                  strides=strides,
                  padding='same',
                  kernel_initializer='he_normal',
                  kernel_regularizer=l2(1e-4))

    x = inputs
    if conv_first:
        x = conv(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
    else:
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
        x = conv(x)
    return x


def resnet_v1(input_shape, depth, num_classes=100):
    """ResNet Version 1 Model builder [a]

    Stacks of 2 x (3 x 3) Conv2D-BN-ReLU
    Last ReLU is after the shortcut connection.
    At the beginning of each stage, the feature map size is halved (downsampled)
    by a convolutional layer with strides=2, while the number of filters is
    doubled. Within each stage, the layers have the same number filters and the
    same number of filters.
    Features maps sizes:
    stage 0: 32x32, 16
    stage 1: 16x16, 32
    stage 2:  8x8,  64
    The Number of parameters is approx the same as Table 6 of [a]:
    ResNet20 0.27M
    ResNet32 0.46M
    ResNet44 0.66M
    ResNet56 0.85M
    ResNet110 1.7M

    # Arguments
        input_shape (tensor): shape of input image tensor
        depth (int): number of core convolutional layers
        num_classes (int): number of classes (CIFAR10 has 10)

    # Returns
        model (Model): Keras model instance
    """
    if (depth - 2) % 6 != 0:
        raise ValueError('depth should be 6n+2 (eg 20, 32, 44 in [a])')
    # Start model definition.
    num_filters = 16
    num_res_blocks = int((depth - 2) / 6)

    inputs = Input(shape=input_shape)
    x = resnet_layer(inputs=inputs)
    # Instantiate the stack of residual units
    for stack in range(3):
        for res_block in range(num_res_blocks):
            strides = 1
            if stack > 0 and res_block == 0:  # first layer but not first stack
                strides = 2  # downsample
            y = resnet_layer(inputs=x,
                             num_filters=num_filters,
                             strides=strides)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters,
                             activation=None)
            if stack > 0 and res_block == 0:  # first layer but not first stack
                # linear projection residual shortcut connection to match
                # changed dims
                x = resnet_layer(inputs=x,
                                 num_filters=num_filters,
                                 kernel_size=1,
                                 strides=strides,
                                 activation=None,
                                 batch_normalization=False)
            x = keras.layers.add([x, y])
            x = Activation('relu')(x)
        num_filters *= 2

    # Add classifier on top.
    # v1 does not use BN after last shortcut connection-ReLU
    x = AveragePooling2D(pool_size=8)(x)
    y = Flatten()(x)
    outputs = Dense(num_classes,
                    activation='softmax',
                    kernel_initializer='he_normal')(y)

    # Instantiate model.
    model = Model(inputs=inputs, outputs=outputs)
    return model


def resnet_v2(input_shape, depth, num_classes=10):
    """ResNet Version 2 Model builder [b]

    Stacks of (1 x 1)-(3 x 3)-(1 x 1) BN-ReLU-Conv2D or also known as
    bottleneck layer
    First shortcut connection per layer is 1 x 1 Conv2D.
    Second and onwards shortcut connection is identity.
    At the beginning of each stage, the feature map size is halved (downsampled)
    by a convolutional layer with strides=2, while the number of filter maps is
    doubled. Within each stage, the layers have the same number filters and the
    same filter map sizes.
    Features maps sizes:
    conv1  : 32x32,  16
    stage 0: 32x32,  64
    stage 1: 16x16, 128
    stage 2:  8x8,  256

    # Arguments
        input_shape (tensor): shape of input image tensor
        depth (int): number of core convolutional layers
        num_classes (int): number of classes (CIFAR10 has 10)

    # Returns
        model (Model): Keras model instance
    """
    if (depth - 2) % 9 != 0:
        raise ValueError('depth should be 9n+2 (eg 56 or 110 in [b])')
    # Start model definition.
    num_filters_in = 16
    num_res_blocks = int((depth - 2) / 9)

    inputs = Input(shape=input_shape)
    # v2 performs Conv2D with BN-ReLU on input before splitting into 2 paths
    x = resnet_layer(inputs=inputs,
                     num_filters=num_filters_in,
                     conv_first=True)

    # Instantiate the stack of residual units
    for stage in range(3):
        for res_block in range(num_res_blocks):
            activation = 'relu'
            batch_normalization = True
            strides = 1
            if stage == 0:
                num_filters_out = num_filters_in * 4
                if res_block == 0:  # first layer and first stage
                    activation = None
                    batch_normalization = False
            else:
                num_filters_out = num_filters_in * 2
                if res_block == 0:  # first layer but not first stage
                    strides = 2    # downsample

            # bottleneck residual unit
            y = resnet_layer(inputs=x,
                             num_filters=num_filters_in,
                             kernel_size=1,
                             strides=strides,
                             activation=activation,
                             batch_normalization=batch_normalization,
                             conv_first=False)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters_in,
                             conv_first=False)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters_out,
                             kernel_size=1,
                             conv_first=False)
            if res_block == 0:
                # linear projection residual shortcut connection to match
                # changed dims
                x = resnet_layer(inputs=x,
                                 num_filters=num_filters_out,
                                 kernel_size=1,
                                 strides=strides,
                                 activation=None,
                                 batch_normalization=False)
            x = keras.layers.add([x, y])

        num_filters_in = num_filters_out

    # Add classifier on top.
    # v2 has BN-ReLU before Pooling
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = AveragePooling2D(pool_size=8)(x)
    y = Flatten()(x)
    outputs = Dense(num_classes,
                    activation='softmax',
                    kernel_initializer='he_normal')(y)

    # Instantiate model.
    model = Model(inputs=inputs, outputs=outputs)
    return model


if version == 2:
    model = resnet_v2(input_shape=input_shape, depth=depth)
else:
    model = resnet_v1(input_shape=input_shape, depth=depth)

model.compile(loss='categorical_crossentropy',
              optimizer=Adam(lr=lr_schedule(0)),
              metrics=['accuracy'])
model.summary()
print(model_type)

# Prepare model model saving directory.
save_dir = os.path.join(os.getcwd(), 'saved_models')
model_name = 'cifar10_%s_model.{epoch:03d}.h5' % model_type
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)
filepath = os.path.join(save_dir, model_name)

# Prepare callbacks for model saving and for learning rate adjustment.
checkpoint = ModelCheckpoint(filepath=filepath,
                             monitor='val_acc',
                             verbose=1,
                             save_best_only=True)

lr_scheduler = LearningRateScheduler(lr_schedule)

lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),
                               cooldown=0,
                               patience=5,
                               min_lr=0.5e-6)

callbacks = [checkpoint, lr_reducer, lr_scheduler]

# Run training, with or without data augmentation.
if not data_augmentation:
    print('Not using data augmentation.')
    model.fit(x_train, y_train,
              batch_size=batch_size,
              epochs=epochs,
              validation_data=(x_test, y_test),
              shuffle=True,
              callbacks=callbacks)
else:
    print('Using real-time data augmentation.')
    # This will do preprocessing and realtime data augmentation:
    datagen = ImageDataGenerator(
        # set input mean to 0 over the dataset
        featurewise_center=False,
        # set each sample mean to 0
        samplewise_center=False,
        # divide inputs by std of dataset
        featurewise_std_normalization=False,
        # divide each input by its std
        samplewise_std_normalization=False,
        # apply ZCA whitening
        zca_whitening=False,
        # epsilon for ZCA whitening
        zca_epsilon=1e-06,
        # randomly rotate images in the range (deg 0 to 180)
        rotation_range=0,
        # randomly shift images horizontally
        width_shift_range=0.1,
        # randomly shift images vertically
        height_shift_range=0.1,
        # set range for random shear
        shear_range=0.,
        # set range for random zoom
        zoom_range=0.,
        # set range for random channel shifts
        channel_shift_range=0.,
        # set mode for filling points outside the input boundaries
        fill_mode='nearest',
        # value used for fill_mode = "constant"
        cval=0.,
        # randomly flip images
        horizontal_flip=True,
        # randomly flip images
        vertical_flip=False,
        # set rescaling factor (applied before any other transformation)
        rescale=None,
        # set function that will be applied on each input
        preprocessing_function=None,
        # image data format, either "channels_first" or "channels_last"
        data_format=None,
        # fraction of images reserved for validation (strictly between 0 and 1)
        validation_split=0.0)

    # Compute quantities required for featurewise normalization
    # (std, mean, and principal components if ZCA whitening is applied).
    datagen.fit(x_train)

    # Fit the model on the batches generated by datagen.flow().
    #model.fit_generator(datagen.flow(x_train, y_train, batch_size=batch_size),
                       # validation_data=(x_test, y_test),
                       # epochs=epochs, verbose=1, workers=4,
                        #callbacks=callbacks)

# Score trained model.
#scores = model.evaluate(x_test, y_test, verbose=1)
#print('Test loss:', scores[0])
#print('Test accuracy:', scores[1])

x_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples
y_train shape: (50000, 1)
Learning rate:  0.001
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 32, 32, 3)]  0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 32, 32, 16)   448         ['input_1[0][0]']                
                                                                                                  
 batch_normalization (BatchNorm  (None, 32, 32, 16)  64          ['conv2d[0][0]']                 
 alization)                                                                                       
                                                                        

 batch_normalization_9 (BatchNo  (None, 32, 32, 16)  64          ['conv2d_9[0][0]']               
 rmalization)                                                                                     
                                                                                                  
 activation_9 (Activation)      (None, 32, 32, 16)   0           ['batch_normalization_9[0][0]']  
                                                                                                  
 conv2d_10 (Conv2D)             (None, 32, 32, 16)   2320        ['activation_9[0][0]']           
                                                                                                  
 batch_normalization_10 (BatchN  (None, 32, 32, 16)  64          ['conv2d_10[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 add_4 (Ad

 conv2d_19 (Conv2D)             (None, 16, 16, 32)   4640        ['activation_18[0][0]']          
                                                                                                  
 batch_normalization_19 (BatchN  (None, 16, 16, 32)  128         ['conv2d_19[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_19 (Activation)     (None, 16, 16, 32)   0           ['batch_normalization_19[0][0]'] 
                                                                                                  
 conv2d_20 (Conv2D)             (None, 16, 16, 32)   9248        ['activation_19[0][0]']          
                                                                                                  
 conv2d_21 (Conv2D)             (None, 16, 16, 32)   544         ['activation_18[0][0]']          
          

                                                                  'batch_normalization_28[0][0]'] 
                                                                                                  
 activation_28 (Activation)     (None, 16, 16, 32)   0           ['add_13[0][0]']                 
                                                                                                  
 conv2d_30 (Conv2D)             (None, 16, 16, 32)   9248        ['activation_28[0][0]']          
                                                                                                  
 batch_normalization_29 (BatchN  (None, 16, 16, 32)  128         ['conv2d_30[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_29 (Activation)     (None, 16, 16, 32)   0           ['batch_normalization_29[0][0]'] 
          

 batch_normalization_38 (BatchN  (None, 8, 8, 64)    256         ['conv2d_39[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 add_18 (Add)                   (None, 8, 8, 64)     0           ['conv2d_40[0][0]',              
                                                                  'batch_normalization_38[0][0]'] 
                                                                                                  
 activation_38 (Activation)     (None, 8, 8, 64)     0           ['add_18[0][0]']                 
                                                                                                  
 conv2d_41 (Conv2D)             (None, 8, 8, 64)     36928       ['activation_38[0][0]']          
                                                                                                  
 batch_nor

 conv2d_50 (Conv2D)             (None, 8, 8, 64)     36928       ['activation_47[0][0]']          
                                                                                                  
 batch_normalization_48 (BatchN  (None, 8, 8, 64)    256         ['conv2d_50[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 add_23 (Add)                   (None, 8, 8, 64)     0           ['activation_46[0][0]',          
                                                                  'batch_normalization_48[0][0]'] 
                                                                                                  
 activation_48 (Activation)     (None, 8, 8, 64)     0           ['add_23[0][0]']                 
                                                                                                  
 conv2d_51

C:\Users\Student\.conda\envs\abc\lib\site-packages\keras\optimizer_v2\adam.py:105: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


                                                                                                  
 conv2d_55 (Conv2D)             (None, 8, 8, 64)     36928       ['activation_52[0][0]']          
                                                                                                  
 batch_normalization_53 (BatchN  (None, 8, 8, 64)    256         ['conv2d_55[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_53 (Activation)     (None, 8, 8, 64)     0           ['batch_normalization_53[0][0]'] 
                                                                                                  
 conv2d_56 (Conv2D)             (None, 8, 8, 64)     36928       ['activation_53[0][0]']          
                                                                                                  
 batch_nor

In [4]:
scores = model.evaluate(x_test, y_test, verbose=1)
print('Test loss:', scores[0])
print('Test accuracy:', scores[1])

313/313 [==============================] - 6s 12ms/step - loss: 1.7128 - accuracy: 0.6996
Test loss: 1.7127923965454102
Test accuracy: 0.6995999813079834


In [4]:
#model.save_weights('ResNet56_100.ckpt')

In [3]:
model.load_weights('ResNet56_100.ckpt')

# Truncation

In [5]:
import numpy as np
import os
from codecs import decode
import struct


#os.environ["CUDA_VISIBLE_DEVICES"] = "-1" #Disables GPU acceleration

global count_total_bits_truncated
count_total_bits_truncated = 0

def read_args():
    f = open('args.txt')
    args = f.read().split('\n')
    f.close()
    return args


def truncate_weights(model, args):
    def bin_truncation(binary_string, n_bits):
        if n_bits == 0: #Error Condition avoidance
            return binary_string

        #fill truncation values
        result = binary_string[:-n_bits] + '1'
        while (len(result) < 32):
            result = result + '0'

        return (result)

    def bin_to_float(b):
        """ Convert binary string to a float. """
        b = b + '00000000000000000000000000000000'  # convert back to 64 bit
        bf = int_to_bytes(int(b, 2), 8)  # 8 bytes needed for IEEE 754 binary64.
        return struct.unpack('>d', bf)[0]

    def int_to_bytes(n, length):  # Helper function
        """ Int/long to byte string.

            Python 3.2+ has a built-in int.to_bytes() method that could be used
            instead, but the following works in earlier versions including 2.x.
        """
        return decode('%%0%dx' % (length << 1) % n, 'hex')[-length:]

    def float_to_bin(value):  # For testing.
        """ Convert float to 32-bit binary string. """
        [d] = struct.unpack(">Q", struct.pack(">d", value))
        return '{:064b}'.format(d)[:32]

    def truncation_process(weight, n_bits):
        a = float_to_bin(weight)
        b = bin_truncation(a, n_bits)
        return bin_to_float(b)

    def recursive_truncate(node, n_bits):
        #if list, dig deeper in recursion
        if 'list' in str(type(node)):
            for count, i in enumerate(node):
                node[count] = recursive_truncate(node[count], n_bits)
            return node
        #if ndarray, dig deeper in recursion
        elif 'ndarray' in str(type(node)):
            for count, i in enumerate(node):
                node[count] = recursive_truncate(node[count], n_bits)
            return node
        #finally, if single weight. truncate and return up recusion
        else:
            global count_total_bits_truncated
            count_total_bits_truncated = count_total_bits_truncated + n_bits
            return truncation_process(node, n_bits)

    def truncation_handler(layer, n_bits):
        def convolution_handler(layer):
            weights = layer.get_weights()
            print('CNN truncation: n_bits = ', n_bits)
            new_weights = recursive_truncate(weights, n_bits)
            return new_weights

        def dense_handler(layer):
            weights = layer.get_weights()
            print('ANN truncation: n_bits = ', n_bits)
            new_weights = recursive_truncate(weights, n_bits)
            return new_weights


        s = str(type(layer))

        if 'Conv2D' in s:
            print('Convolution')
            return convolution_handler(layer)

        elif 'Dense' in s:
            print('DENSE')
            return dense_handler(layer)

        else:
            print(type(layer))
            print(layer.get_weights())
            print(s)
            print('LAYER UNEXPECTED')
            exit()

    def check_skip_layers(layer):
        s = str(type(layer))
        if 'convolutional.UpSampling2D' in s:
            print('action: PASS - up sampling')
            return True

        elif 'InputLayer' in s:
            print('action: PASS - InputLayer')
            return True
        elif 'ZeroPadding2D' in s:
            print('action: PASS - ZeroPadding2D')
            return True

        elif 'MaxPooling2D' in s:
            print('actoin: PASS - MaxPooling2D')
            return True

        elif 'Dropout' in s:
            print('action: PASS - dropout')
            return True

        elif 'Flatten' in s:
            print('action: PASS - Flatten')
            return True
        elif 'ReLU' in s:
            print('action: PASS - ReLU')
            return True
        elif 'Activation' in s:
            print('action: PASS - Activation')
            return True
        elif 'UpSampling' in s:
            print('action: PASS - UpSampiling')
            return True
        elif 'Add' in s:
            print('action: PASS - Add')
            return True
        elif 'AveragePooling2D' in s:
            print('action: PASS - AveragePooling2D')
            return True
        elif 'BatchNormalization' in s:
            print('BATCH NORMALIZATION')
            return True
        else:
            return False


    #begin truncation
    layerTruncationIndex = 0

    print('Model has Layers = ', len(model.layers))
    for count, layer in enumerate(model.layers):
        if check_skip_layers(layer):
            pass
        else:
            #get n_bits via a global counter
            n_bits = int(args[layerTruncationIndex])
            new_layer = truncation_handler(layer, n_bits)
            model.layers[count].set_weights(new_layer)
            layerTruncationIndex = layerTruncationIndex + 1

        print('*************************************')


    return model


def test_model(model, x_train, y_train, x_test, y_test, args, file_name='Model'):

    print('Begin Testing....')
    total_correct = 0
    total_possible = y_train.shape[0] + y_test.shape[0]

    predictions = np.argmax(model.predict(x_train), axis=-1)
    for i in range(predictions.shape[0]):
        if (predictions[i] == np.argmax(y_train[i])):
            total_correct = total_correct + 1

    predictions = np.argmax(model.predict(x_test), axis=-1)

    for i in range(predictions.shape[0]):
        if (predictions[i] == np.argmax(y_test[i])):
            total_correct = total_correct + 1

    print('Total Correct: ', total_correct)
    print('Total Possibl: ', total_possible)

    print('Accuracy = ', (total_correct/total_possible))
    print(count_total_bits_truncated)

    #Save all of the important information into csv file
    for i in args:
        file_name = file_name + '_' + str(i)
    file_name = file_name + '.csv'

    f = open(file_name, "a")
    for i in args:
        f.write(str(i) + ',')
    f.write(str(total_correct) + ',' + str(total_possible) + ',' + str(count_total_bits_truncated))
    f.close()

In [7]:
model.save('model.h5') 
#import tensorflow
#import Models
#import keras
#import numpy as np
import os
#import TruncationEmulator
#from tensorflow.keras.optimizers import RMSprop

def main():
    #model = Models.load_model_resNet50(input_shape=(32, 32, 3))
    #model = Models.load_model_alexnet(input_shape=(32, 32, 3))
    #x_train, y_train, x_test, y_test = Models.load_data()

    #model.compile(loss='categorical_crossentropy',
                 # optimizer=tensorflow.keras.optimizers.RMSprop(learning_rate=2e-5),
                 # metrics=['accuracy'])

    model.load_weights('model.h5') 
    #model.summary()
    model.evaluate(x_test, y_test, verbose=1) 
    variables = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15, 16,17,18,19,20,21,22,23,24,25,26,27,28,29,30, 31]
    #variables=[0,16,31]
    for i1 in variables:
        args = [ i1, i1, i1, i1,i1, i1, i1, i1, i1, i1, i1,i1,i1, i1, i1,i1,i1, i1, i1,i1, i1, i1, i1,i1, i1, i1, i1, i1, i1, i1,i1,i1, i1, i1,i1,i1, i1, i1, i1, i1, i1, i1,i1, i1, i1, i1, i1, i1, i1,i1,i1, i1, i1,i1,i1, i1,i1,0] #spesific to AlexNet, makes it so the final output layer is never truncated

        truncate_model = truncate_weights(model, args)

        test_model(truncate_model, x_test, y_test, x_test, y_test, args, 'model')



if __name__ == "__main__":
    main()

313/313 [==============================] - 4s 12ms/step - loss: 1.7128 - accuracy: 0.6996
Model has Layers =  198
action: PASS - InputLayer
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
***

*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
Convolution
CNN truncation: n_bits =  0
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  0
********

*************************************
Convolution
CNN truncation: n_bits =  1
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  1
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  1
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  1
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  1
********

ANN truncation: n_bits =  0
*************************************
Begin Testing....
Total Correct:  13990
Total Possibl:  20000
Accuracy =  0.6995
852992
Model has Layers =  198
action: PASS - InputLayer
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
***

*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
Convolution
CNN truncation: n_bits =  2
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  2
********

...

CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZA

*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  29
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
action: PASS - AveragePooling2D
*************************************
action: PASS - Flatten
*************************************
DENSE
ANN truncation: n_bits =  0
*************************************
Begin Testing....
Total Correct:  200
Total Possibl:  20000
Accuracy =  0.01
371051520
Model has Layers =  198
action: PA

*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  30
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  30
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  30
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  30
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
******************

CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
Convolution
CNN

*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Activation
*************************************
Convolution
CNN truncation: n_bits =  31
*************************************
BATCH NORMALIZATION
*************************************
action: PASS - Add
*************************************
action: PASS - Activation
*************************************
action: PASS - AveragePooling2D
*************************************
action: PASS - Flatten
******************************